In [31]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI 

load_dotenv(override=True)
api_key=os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("API key looks good so far")
elif api_key.strip() != api_key:
    print("API KEY LOOKS SO SAME NEAR")
else:
    print("Right way to api key")





Right way to api key


In [32]:
Model ="openai/gpt-4o"

liks=fetch_website_links("https://github.com/jamilhossain1997")

In [33]:
links_system_prompt="""
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://github.com/jamilhossain1997/jamilhossain1997"},
        {"type": "careers page", "url": "https://drive.google.com/file/d/1146vlZL9r6d-uqBFrTEJohseYbi3jxsw/view"}
    ]
}
"""

In [34]:
from openai import OpenAI

base_url="https://openrouter.ai/api/v1"
client=OpenAI(base_url=base_url,api_key=api_key)

response = client.chat.completions.create(
    model=Model,
    messages=[
        {"role":"system","content":links_system_prompt}
    ],
        max_tokens=1000

)

print(response.choices[0].message.content)

{
    "links": [
        {"type": "about page", "url": "https://github.com/jamilhossain1997/jamilhossain1997"},
        {"type": "careers page", "url": "https://drive.google.com/file/d/1146vlZL9r6d-uqBFrTEJohseYbi3jxsw/view"}
    ]
}


In [35]:
def user_url_link_prompt(url):
    links = fetch_website_links(url)

    user_prompt = f"""
For GitHub links:

- Include GitHub repositories if they demonstrate Software Engineering,
  AI Engineering, LLM Engineering, AI agents, RAG, or related technical work.

- Identify the link type as "Software Engineering GitHub",
  "AI Engineering GitHub", or another appropriate category.

Here are the links found on the website {url}:

{chr(10).join(links)}
"""

    return user_prompt

In [36]:
print(user_url_link_prompt("https://github.com/jamilhossain1997"))


For GitHub links:

- Include GitHub repositories if they demonstrate Software Engineering,
  AI Engineering, LLM Engineering, AI agents, RAG, or related technical work.

- Identify the link type as "Software Engineering GitHub",
  "AI Engineering GitHub", or another appropriate category.

Here are the links found on the website https://github.com/jamilhossain1997:

#start-of-content
/
/login?return_to=https%3A%2F%2Fgithub.com%2Fjamilhossain1997
https://github.com/features/copilot
https://github.com/features/ai/github-app
https://github.com/mcp
https://github.com/features/actions
https://github.com/features/codespaces
https://github.com/features/issues
https://github.com/features/code-review
https://github.com/features/code-quality
https://github.com/security/advanced-security
https://github.com/security/advanced-security/code-security
https://github.com/security/advanced-security/secret-protection
https://github.com/why-github
https://docs.github.com
https://github.blog
https://github

In [37]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": "You respond only in valid JSON format."},
            {"role": "user", "content": user_url_link_prompt(url)}
        ],
        response_format={'type': 'json_object'},
        max_tokens=1000
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    return links

In [38]:
select_relevant_links("https://github.com/jamilhossain1997")

{'GitHubLinks': [{'link': '/jamilhossain1997/ecomwebapi',
   'type': 'Software Engineering GitHub'},
  {'link': '/jamilhossain1997/ecomapi', 'type': 'Software Engineering GitHub'},
  {'link': '/jamilhossain1997/fristCodeTest',
   'type': 'Software Engineering GitHub'},
  {'link': '/jamilhossain1997/finalCodeTest',
   'type': 'Software Engineering GitHub'},
  {'link': '/jamilhossain1997/bdfrontend',
   'type': 'Software Engineering GitHub'},
  {'link': '/jamilhossain1997/bdfrontend1',
   'type': 'Software Engineering GitHub'}]}

In [39]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for category, links in relevant_links.items():

        for link in links:
            link_type = link.get("category", category)
            link_url = link.get("link")

            if not link_url:
                continue

            result += f"\n\n### Link: {link_type}\n"
            result += fetch_website_contents(link_url)

    return result

In [40]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [41]:
def get_brochure_user_prompt(company_name,url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [42]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

"\n    You are looking at a company called: HuggingFace\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    ## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nconvaiinnovations/laya\nUpdated\n2 days ago\n•\n3.84k\nQwen/Qwen-Image-2.1\nUpdated\n6 days ago\n•\n

In [46]:
def create_brocha(company_name, url):

    response = client.chat.completions.create(
        model=Model,
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_user_prompt(company_name, url)
            }
        ],
        max_tokens=2000
    )

    result = response.choices[0].message.content

    display(Markdown(result))

In [47]:
create_brocha("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

Welcome to Hugging Face, where innovation meets collaboration in the thriving world of artificial intelligence. We are the dedicated platform empowering the global machine learning community to create, discover, and share cutting-edge models, datasets, and applications. 

## About Us
Hugging Face is more than just a tech company; it is a community that plays a pivotal role in shaping the future of AI. Our collaborative platform hosts over 2 million models, 1 million applications, and 500,000 datasets, setting the stage for AI enthusiasts, researchers, and developers to collaborate and innovate together.

## What We Offer
- **Models & Datasets:** Explore and leverage our extensive library of models and datasets. Stay ahead of the curve with trending models like `Qwen/Qwen-Image-2.1` and `XingChen-AGI/Xing4.0-29B-A4B`.
  
- **Applications & Spaces:** Engage with a vast array of applications ranging from image generation to decision-making frameworks. Check out highlight projects such as `Animate a picture into a short video` or `Generate and edit images from text prompts`.

- **Team & Enterprise Solutions:** Hugging Face offers tailored enterprise support and solutions including inference providers, endpoints, and storage options to suit various business needs.

## Our Community
The heartbeat of Hugging Face lies in our vibrant, inclusive AI community. We provide diverse platforms for interaction such as forums, Discord, and GitHub, welcoming collaboration and allowing people to learn collectively. Our community is enabled to share insights, participate in daily discussions, and contribute to the world of machine learning.

## Careers and Work Culture
At Hugging Face, we are always looking for passionate individuals to join us on our mission. We offer a work environment that is both challenging and rewarding, encouraging creativity, curiosity, and continuous learning. As part of our team, you will be at the forefront of technological advancement, helping to pioneer state-of-the-art solutions in AI.

## Join Us
Whether you are a researcher, developer, investor, or aspiring AI enthusiast, Hugging Face is your gateway to contribute towards building the future of AI. Sign up for free, delve into a wealth of resources, and become an integral part of our revolutionary journey.

Visit our website to learn more about our offerings and sign up today to be a part of the AI-driven change. Together, let's innovate, inspire, and influence the future of artificial intelligence.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(country_name,url):
    stream = client.chat.completions.create(
        model=Model,
        messages=[
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content" : get_brochure_user_prompt(country_name,url)}
        ],
        max_tokens=3000,
        stream=True
    )
    
    response =""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id)

In [53]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

## Welcome to Hugging Face

Hugging Face is at the forefront of the artificial intelligence community, dedicated to building the future of technology. As a collaborative platform, we aim to bring the machine learning community together to develop, share, and innovate on models, datasets, and applications.

### What We Offer

- **Models**: Access over 2 million machine learning models. Our platform allows you to browse, collaborate, and improve diverse models that cater to various AI tasks.
  
- **Datasets**: Discover and utilize a vast collection of over 500,000 datasets to refine and train your models for enhanced performance.
  
- **Spaces**: Explore and run more than 1 million applications. Experiment and deploy AI solutions capable of transforming data into actionable insights.

### Key Features

- **Collaborative Environment**: A dynamic space for AI enthusiasts and professionals to host and collaborate on unlimited public models and datasets.
  
- **Enterprise Solutions**: Tailored solutions through our Team & Enterprise plans, designed to support and scale machine learning endeavors in larger organizations.
  
- **Community Tools**: Engage with our extensive community through platforms like Discord and GitHub, share ideas, and drive collective learning and growth.

### Our Community

Join a vibrant and inclusive community. Hugging Face is not just a company; it's a collective of developers, researchers, and enthusiasts passionate about pushing the boundaries of what's possible with machine learning and AI.

### Career Opportunities

Hugging Face is continually expanding, seeking innovative minds eager to shape the future of artificial intelligence. We value creativity, collaboration, and a commitment to learning. Explore opportunities with us to embark on a career that will both challenge and inspire you.

### Connect with Us

- **Website**: Visit [our website](https://huggingface.co) to explore models, datasets, and the latest innovations in AI.
- **Community Platforms**: Join our conversations on [GitHub](https://github.com/huggingface), [Discord](https://discord.com/invite/huggingface), and follow our updates and articles on the Hugging Face blog.

Be a part of the AI revolution with Hugging Face, and together, let’s build a future powered by innovation and collaboration.